In [15]:
# Cài đặt các thư viện cần thiết cho HybridRAG
# Chạy ô này nếu chưa cài các package bên dưới
!pip install llama-index weaviate-client huggingface-hub pymupdf llama-index-embeddings-huggingface gradio

c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [13]:
import os

current_dir = os.getcwd()
print("Current working directory:", current_dir)

Current working directory: c:\Users\Admin\Desktop\Paraline_2\Rag_demo_paraline\use_case\hybrid_rag


In [16]:
import fitz  # PyMuPDF

pdf_path = "..//../data_sample/A 2022 Comptia Security+ Guide to Network Security Fundamentals by Mark Ciampa (z-lib.org) (1).pdf"
doc = fitz.open(pdf_path)

toc = doc.get_toc()  # Lấy table of contents, mỗi item: [level, title, page_number]
for level, title, page in toc:
    
    print(f"Level: {level}, Title: {title}, Start Page: {page}")


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


Level: 1, Title: Brief Contents, Start Page: 7
Level: 1, Title: Table of Contents, Start Page: 8
Level: 1, Title: Introduction, Start Page: 13
Level: 1, Title: Part 1: Security Fundamentals, Start Page: 23
Level: 2, Title: Module 1: Introduction To Security, Start Page: 25
Level: 3, Title: What Is Information Security?������������������������������������������������������������������������, Start Page: 27
Level: 3, Title: Who Are the Threat Actors?������������������������������������������������������������������, Start Page: 29
Level: 3, Title: Vulnerabilities and Attacks��������������������������������������������������������������������, Start Page: 33
Level: 3, Title: Summary, Start Page: 44
Level: 3, Title: Key Terms, Start Page: 45
Level: 3, Title: Review Questions, Start Page: 46
Level: 3, Title: Case Projects, Start Page: 52
Level: 2, Title: Module 2: Threat Management and Cybersecurity Resources, Start Page: 55
Level: 3, Title: Penetration Testing������������������������������

In [17]:
import fitz  # PyMuPDF
import re
from llama_index.core import Document


def clean_text(s: str) -> str:
    """Làm sạch text để tránh lỗi UnicodeEncodeError"""
    if not isinstance(s, str):
        return ""
    s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")
    s = re.sub(r"[\x00-\x1F\x7F]", " ", s)
    return " ".join(s.split())


def clean_title(title: str) -> str:
    """Mọi level: chỉ lấy phần trước dấu '\'"""
    if not isinstance(title, str) or not title.strip():
        return "Untitled"
    title = title.split("\\")[0]   # chỉ giữ phần trước dấu "\"
    return clean_text(title).strip()


def chunk_pdf_hierarchical(pdf_path: str):
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()
    max_page = doc.page_count
    documents = []

    if not toc:
        text = "\n".join([doc[p].get_text("text") for p in range(max_page)])
        return [
            Document(
                text=clean_text(text),
                metadata={
                    "section": "Full Document",
                    "start_page": 1,
                    "end_page": max_page,
                    "level": 0,
                },
            )
        ]

    parents = {}
    for i, (level, raw_title, start_page) in enumerate(toc):
        title = clean_title(raw_title)

        if i + 1 < len(toc):
            end_page = toc[i + 1][2] - 1
        else:
            end_page = max_page

        # cập nhật cha
        parents[level] = title
        for l in list(parents.keys()):
            if l > level:
                parents.pop(l)

        # metadata gồm toàn bộ hierarchy
        metadata = {f"level_{l}": parents[l] for l in sorted(parents.keys())}
        metadata.update(
            {
                "section": title,
                "start_page": start_page,
                "end_page": end_page,
                "level": level,
            }
        )

        # lấy text của đoạn
        text = "\n".join([doc[p].get_text("text") for p in range(start_page - 1, end_page)])
        documents.append(Document(text=clean_text(text), metadata=metadata))

    return documents


In [18]:
# --- Example usage ---
pdf_path = "../Security/A 2022 Comptia Security+ Guide to Network Security Fundamentals by Mark Ciampa (z-lib.org) (1).pdf"
chunks = chunk_pdf_hierarchical(pdf_path)
print(f"Extracted {len(chunks)} chunks")
print("Example metadata:", chunks[5].metadata)

Extracted 130 chunks
Example metadata: {'level_1': 'Part 1: Security Fundamentals', 'level_2': 'Module 1: Introduction To Security', 'level_3': 'What Is Information Security?', 'section': 'What Is Information Security?', 'start_page': 27, 'end_page': 28, 'level': 3}


In [17]:
from llama_index.core.node_parser import HierarchicalNodeParser

def split_chunks_with_llama(documents):
    parser = HierarchicalNodeParser.from_defaults(
        chunk_sizes=[1500, 500, 200]
    )

    results = []
    for i, doc in enumerate(documents, start=1):
        # lấy nodes từ từng Document
        nodes = parser.get_nodes_from_documents([doc])

        for j, node in enumerate(nodes, start=1):
            # merge metadata: giữ TOC metadata gốc + metadata node parser sinh ra
            merged_meta = {**doc.metadata, **node.metadata}

            results.append({
                "parent_chunk": i,
                "sub_chunk": j,
                "text": node.text,
                "metadata": merged_meta
            })

    return results


# --- Example usage ---
final_chunks = split_chunks_with_llama(chunks)  # chunks là list Document
print(f"Split into {len(final_chunks)} hierarchical sub-chunks")

for fc in final_chunks[:3]:
    print(fc["metadata"])
    print(fc["text"][:120], "...\n---")


Split into 5555 hierarchical sub-chunks
{'level_1': 'Brief Contents', 'section': 'Brief Contents', 'start_page': 7, 'end_page': 7, 'level': 1}
BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
---
{'level_1': 'Brief Contents', 'section': 'Brief Contents', 'start_page': 7, 'end_page': 7, 'level': 1}
BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
---
{'level_1': 'Brief Contents', 'section': 'Brief Contents', 'start_page': 7, 'end_page': 7, 'level': 1}
BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
---


In [122]:
final_chunks[999]

{'parent_chunk': 28,
 'sub_chunk': 44,
 'text': 'In this project, you explore different ransomware sites. 1. Open your web browser and enter the URL www.nomoreransom.org (if you are not able to access this site open a search engine and search for “Nomoreransom.org”). 2. Click the No button. 3. Read through the Prevention Advice. Do you think it is helpful? 4. Click Crypto Sheriff. How could this be useful to a user who has suffered a ransomware infection? 5. Click Ransomware: Q&A. Read through the information. Which statements would you agree with? Which statements would you disagree with?',
 'metadata': {'level_1': 'Part 2: Endpoint Security',
  'level_2': 'Module 3: Threats and Attacks on Endpoints',
  'level_3': 'Review Questions',
  'section': 'Review Questions',
  'start_page': 110,
  'end_page': 114,
  'level': 3},
 'embedding': [-0.005149153061211109,
  -0.006778764072805643,
  -0.011947466991841793,
  -0.03821266442537308,
  0.07553940266370773,
  -0.0785190612077713,
  -0.0449

In [120]:
!docker compose up -d

d:\anaconda3\envs\llm_pipeline\Lib\site-packages\weaviate\warnings.py:292: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\Admin\AppData\Local\Temp\ipykernel_23156\1342092772.py:12: ResourceWarning: unclosed <socket.socket fd=8156, family=23, type=1, proto=0, laddr=('::1', 51986, 0, 0), raddr=('::1', 8080, 0, 0)>
  security = client.collections.create(


In [19]:
import weaviate
import weaviate.classes as wvc
client = weaviate.connect_to_local()

In [ ]:
# Xóa tất cả collection tên 'Security'
try:
    client.collections.delete("Security")
except Exception as e:
    print(f"Không thể xóa collection 'Security': {e}")

# Tạo lại collection 'Security'
security = client.collections.create(
    "Security",
    vector_config=wvc.config.Configure.Vectors.self_provided(),
)


In [20]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import uuid

# --- chọn model embedding Qwen ---
embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")

In [ ]:
# --- chọn collection ---
security = client.collections.get("Security")

# --- batch insert ---
with security.batch.dynamic() as batch:   # dynamic batch: tự động chia nhỏ
    for i, chunk in enumerate(final_chunks[:400], start=1):
        text = chunk["text"]

        # tạo embedding
        embedding = embed_model.get_text_embedding(text)
        chunk["embedding"] = embedding

        # log thông tin embedding
        if i <= 3:  # chỉ in vài cái đầu
            print(f"\n[Chunk {i}]")
            print("Text preview:", text[:120].replace("\n", " "), "...")
            print("Embedding length:", len(embedding))
            print("First 10 dims:", embedding[:10])

        batch.add_object(
            properties={
                "text": text,
                **chunk["metadata"],   # gộp metadata
            },
            vector=embedding,
            uuid=str(uuid.uuid4())
        )

        if i % 100 == 0:
            print(f"Inserted {i} chunks...")

print("✅ Done. Total inserted:", len(final_chunks))



[Chunk 1]
Text preview: BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
Embedding length: 1024
First 10 dims: [0.04021291434764862, -0.004086177796125412, -0.005779475439339876, -0.0515277162194252, 0.08357816934585571, -0.037017665803432465, 0.002998973010107875, 0.042303841561079025, -0.01674053631722927, -0.029778968542814255]

[Chunk 2]
Text preview: BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
Embedding length: 1024
First 10 dims: [0.04021291434764862, -0.004086177796125412, -0.005779475439339876, -0.0515277162194252, 0.08357816934585571, -0.037017665803432465, 0.002998973010107875, 0.042303841561079025, -0.01674053631722927, -0.029778968542814255]

[Chunk 2]
Text preview: BRIEF CONTENTS Introduction IX part 1 SECURITY FUNDAMENTAlS 1 Module 1 Introduction to Security 3 Module 2 Threat Manage ...
Embedding length: 1024
First 1

In [37]:
last_chunk = final_chunks[399]
last_chunk 

{'parent_chunk': 12,
 'sub_chunk': 19,
 'text': 'Write a one-page FAQ about security employment. References 1. Shi, Fleming, “Threat spotlight: Coronavirus-related phishing,” Barracuda, Mar. 26, 2020, accessed Apr. 19, 2020, https://blog.barracuda.com/2020/03/26/threat-spotlight-coronavirus-related-phishing/. 2. “Fake ‘Corona Antivirus’ distributes BlackNET remote administration tool,” MalwareBytes Labs, Mar. 23, 2020, accessed Apr.',
 'metadata': {'level_1': 'Part 1: Security Fundamentals',
  'level_2': 'Module 1: Introduction To Security',
  'level_3': 'Case Projects',
  'section': 'Case Projects',
  'start_page': 52,
  'end_page': 54,
  'level': 3}}

In [21]:
security  = client.collections.get("Security")

c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\Admin\AppData\Local\Temp\ipykernel_8796\2194493129.py:1: ResourceWarning: unclosed <socket.socket fd=2604, family=23, type=1, proto=0, laddr=('::1', 59161, 0, 0), raddr=('::1', 8080, 0, 0)>
  security  = client.collections.get("Security")


In [22]:

bm25_results = security.query.bm25(query="supply chain", limit=3)

for obj in bm25_results.objects:
    print(obj.properties)

{'level_2': 'Module 1: Introduction To Security', 'start_page': 33.0, 'end_page': 43.0, 'text': '• Supply chain. A supply chain is a network that moves a product from the supplier to the customer and is made up of vendors that supply raw material, manufacturers who convert the material into products, warehouses that store products, distribution centers that deliver them to the retailers, and retailers who bring the product to the consumer. Today’s supply chains are global in scope: manufacturers are usually thousands of miles away overseas and not under the direct supervision of the enterprise selling the product. The fact that products move through many steps in the supply chain—and that some steps are not closely supervised—has opened the door for malware to be injected into products during their manufacturing or storage (called supply chain infections).', 'level_1': 'Part 1: Security Fundamentals', 'level': 3.0, 'section': 'Vulnerabilities and Attacks', 'level_3': 'Vulnerabilities a

In [23]:
results = security.query.near_vector(near_vector=embed_model.get_text_embedding("What is supply chain?"), limit=3)

In [24]:
for obj in results.objects:
    print(obj.properties)

{'level_2': 'Module 1: Introduction To Security', 'start_page': 33.0, 'end_page': 43.0, 'text': '• Supply chain. A supply chain is a network that moves a product from the supplier to the customer and is made up of vendors that supply raw material, manufacturers who convert the material into products, warehouses that store products, distribution centers that deliver them to the retailers, and retailers who bring the product to the consumer. Today’s supply chains are global in scope: manufacturers are usually thousands of miles away overseas and not under the direct supervision of the enterprise selling the product. The fact that products move through many steps in the supply chain—and that some steps are not closely supervised—has opened the door for malware to be injected into products during their manufacturing or storage (called supply chain infections).', 'level_1': 'Part 1: Security Fundamentals', 'level_3': 'Vulnerabilities and Attacks', 'section': 'Vulnerabilities and Attacks', '

In [25]:
hybrid_results = security.query.hybrid(
    alpha=0.5,
    query="What is supply chain",
    vector=embed_model.get_text_embedding("What is supply chain"),
    limit=3
)

In [26]:
for object in hybrid_results.objects:
    print(object.properties)

{'start_page': 33.0, 'level_2': 'Module 1: Introduction To Security', 'end_page': 43.0, 'text': '• Supply chain. A supply chain is a network that moves a product from the supplier to the customer and is made up of vendors that supply raw material, manufacturers who convert the material into products, warehouses that store products, distribution centers that deliver them to the retailers, and retailers who bring the product to the consumer. Today’s supply chains are global in scope: manufacturers are usually thousands of miles away overseas and not under the direct supervision of the enterprise selling the product. The fact that products move through many steps in the supply chain—and that some steps are not closely supervised—has opened the door for malware to be injected into products during their manufacturing or storage (called supply chain infections).', 'level_1': 'Part 1: Security Fundamentals', 'level': 3.0, 'section': 'Vulnerabilities and Attacks', 'level_3': 'Vulnerabilities a

In [ ]:
import gradio as gr
from llama_index.llms.openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

# ===============================
# 1. Setup LLM
# ===============================
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = OpenAI(api_key=openai_api_key, model="gpt-4o-mini")

# ===============================
# 2. Memory (simple session history)
# ===============================
chat_history = []

# ===============================
# 3. Conversational query
# ===============================
def conversational_query(message: str):
    # Build context from chat history
    context_text = "\n".join([f"{m['role']}: {m['content']}" for m in chat_history])

    # Example: Hybrid retrieval from Weaviate
    hybrid_results = security.query.hybrid(
        query=message,
        vector=embed_model.get_text_embedding(message),
        alpha=0.1,
        limit=3
    )
    retrieved_text = "\n\n".join([obj.properties.get("text", "")[:300] for obj in hybrid_results.objects])

    # Prompt for LLM
    final_prompt = (
        f"You are a helpful assistant. Answer the user based on the retrieved information below.\n"
        f"Retrieved info:\n{retrieved_text}\n\n"
        f"Conversation so far:\n{context_text}\nUser: {message}\nAssistant:"
    )

    # Get LLM response as string
    response_text = llm.complete(final_prompt)

    # Wrap as dictionary for Gradio Chatbot
    return {"role": "assistant", "content": str(response_text)}

# ===============================
# 4. Gradio chat function
# ===============================
def chat_fn(message, history):
    user_msg = {"role": "user", "content": str(message)}
    assistant_msg = conversational_query(message)

    # Save to session history
    chat_history.append(user_msg)
    chat_history.append(assistant_msg)

    # Update Gradio chatbot history
    history.append(user_msg)
    history.append(assistant_msg)
    return "", history

# ===============================
# 5. Gradio UI
# ===============================
with gr.Blocks() as demo:
    gr.Markdown("## 📚 HybridRAG Conversational Chat")
    chatbot = gr.Chatbot(type="messages", height=500)
    msg = gr.Textbox(label="Your message")
    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot])

    clear_btn = gr.Button("Clear Chat")
    def clear_fn():
        chat_history.clear()
        return []
    clear_btn.click(clear_fn, outputs=[chatbot])

demo.launch(share=True)


c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\websockets\legacy\__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
c:\Users\Admin\anaconda3\envs\hybrid_rag\Lib\site-packages\uvicorn\protocols\websockets\websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
